# Chapter 8 Lab — Pretrained Models and Fine-Tuning

Probes a pretrained masked-LM, fine-tunes a small classification head on a tiny labeled set,
and compares against a from-scratch TF-IDF baseline. Closes with a clean, from-public-weights
extractive-QA reader (not vendored from any third-party framework).

## 1. Zero-shot probing of a pretrained masked-LM

In [ ]:
from transformers import pipeline

fill = pipeline("fill-mask", model="distilbert-base-uncased")
for r in fill("The capital of France is [MASK]."):
    print(r["token_str"], round(r["score"], 3))

## 2. Baseline: TF-IDF + logistic regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

texts = [
    "I loved this movie, fantastic acting", "Terrible plot, waste of time",
    "A wonderful experience from start to finish", "Boring and way too long",
    "Brilliant direction and a great soundtrack", "I regret watching this",
    "One of the best films I've seen this year", "Awful, could not finish it",
    "Highly recommend, superb storytelling", "Disappointing sequel, skip it",
] * 3
labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0] * 3
Xtr, Xte, ytr, yte = train_test_split(texts, labels, test_size=0.3, random_state=0)

vec = TfidfVectorizer().fit(Xtr)
clf = LogisticRegression().fit(vec.transform(Xtr), ytr)
print("Baseline accuracy:", clf.score(vec.transform(Xte), yte))

## 3. Fine-tuning a pretrained encoder on the same data

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch

tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

class ToyDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tok(texts, truncation=True, padding=True, return_tensors="pt")
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

train_ds, eval_ds = ToyDataset(Xtr, ytr), ToyDataset(Xte, yte)
args = TrainingArguments(output_dir="./tmp", num_train_epochs=3, per_device_train_batch_size=8,
                          logging_steps=5, report_to=[])
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds)
trainer.train()
print(trainer.evaluate())

## 4. A clean extractive-QA reader (fine-tuned BERT-style span predictor)

In [ ]:
qa = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")
context = "The Transformer architecture was introduced in the 2017 paper 'Attention Is All You Need'."
print(qa(question="When was the Transformer introduced?", context=context))

## Exercise

Shrink the training set to 4 examples and re-run fine-tuning. How much does accuracy degrade
compared to the from-scratch TF-IDF baseline trained on the same 4 examples? What does that say
about how much of fine-tuning's advantage comes from the labeled data vs. the pretrained
weights?